# 📐 Notebook 3: Metric Computation — BLEU, CIDEr, METEOR, ROUGE-L

**Reads:** `captions_all_models.csv`  
**Writes:** `metrics_all_models.csv`, `metric_summary.csv`

---

### Why compute metrics if we distrust them?
We need to **quantify the failure of metrics**, not avoid them.  
The research claim is: *high metric score ≠ semantic correctness.*  
To prove this we need both the metric scores (here) and human labels (NB04),  
then compare them in NB05+.


In [ ]:
# CELL 1 — Install
import subprocess, sys
for pkg in ['sacrebleu','nltk','rouge-score','pandas','numpy',
            'matplotlib','seaborn','tqdm','scipy','scikit-learn']:
    subprocess.check_call([sys.executable,'-m','pip','install',pkg])
import nltk
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)
nltk.download('punkt',   quiet=True)
print('✅ Done.')

In [ ]:
# CELL 2 — Imports & config
import sys, warnings, random, ast
from pathlib import Path
from typing import List

sys.path.insert(0, str(Path('.')))
import config as cfg

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
from scipy import stats
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from nltk.translate.meteor_score import meteor_score
from rouge_score import rouge_scorer
warnings.filterwarnings('ignore')

SEED   = cfg.SEED
MODELS = cfg.MODELS_TO_RUN
OUTPUT_DIR  = cfg.OUTPUT_DIR
FIGURES_DIR = cfg.FIGURES_DIR
random.seed(SEED); np.random.seed(SEED)

MODEL_CAP_COLS = {
    'blip':'blip_caption','blip2':'blip2_caption',
    'ofa':'ofa_caption','vit_gpt2':'vit_gpt2_caption'
}
BLEU_HIGH_THRESHOLD = 0.20   # used for misalignment analysis in NB04

print('✅ Config ready.')

In [ ]:
# CELL 3 — Load caption DataFrame
cap_path = OUTPUT_DIR / 'captions_all_models.csv'
if not cap_path.exists():
    raise FileNotFoundError('Run 02_caption_generation.ipynb first.')

df = pd.read_csv(cap_path)

# Parse ground_truth_caps (stored as string repr of list)
def parse_caps(val):
    if isinstance(val, list): return val
    try:
        p = ast.literal_eval(str(val))
        return p if isinstance(p, list) else [str(val)]
    except: return [str(val)]

df['gt_caps_parsed'] = df['ground_truth_caps'].apply(parse_caps)
refs_list = df['gt_caps_parsed'].tolist()

print(f'✅ Loaded: {len(df)} rows')
print(f'   Avg GT captions per image: {df["n_gt_captions"].mean():.1f}')
df[['image_id','scenario_category','primary_gt_caption']].head(3)

In [ ]:
print(cfg.MODELS_TO_RUN)

## 📏 Metric Implementations

In [ ]:
# CELL 4 — BLEU (sentence-level, n=1..4)
sf = SmoothingFunction().method1

def compute_bleu_scores(hyps: List[str], refs_list: List[List[str]]):
    results = {f'bleu{n}': [] for n in range(1,5)}
    for hyp, refs in zip(hyps, refs_list):
        hyp_tok  = str(hyp).lower().split() if pd.notna(hyp) else []
        refs_tok = [str(r).lower().split() for r in refs if pd.notna(r)] or [[]]
        for n in range(1,5):
            w = tuple([1/n]*n + [0]*(4-n))
            try:
                s = sentence_bleu(refs_tok, hyp_tok, weights=w, smoothing_function=sf)
            except: s = 0.0
            results[f'bleu{n}'].append(s)
    return results

# Test
t = compute_bleu_scores(['a man on a couch'],[['a person sitting on a sofa']])
print('BLEU test:', {k:f'{v[0]:.3f}' for k,v in t.items()})

In [ ]:
# CELL 5 — CIDEr (lightweight TF-IDF cosine, portable)
from collections import Counter
import math

def compute_cider_scores(hyps, refs_list, n=4):
    def ngrams(tok, n):
        return Counter(tuple(tok[i:i+n]) for i in range(len(tok)-n+1))

    all_ref_ngs = []
    for refs in refs_list:
        for r in refs:
            for ni in range(1, n+1):
                all_ref_ngs.append(ngrams(str(r).lower().split(), ni))
    df_cnt: Counter = Counter()
    for ng in all_ref_ngs:
        for g in set(ng): df_cnt[g] += 1
    N = max(len(all_ref_ngs), 1)
    idf = lambda g: math.log((N+1)/(df_cnt.get(g,0)+1))

    scores = []
    for hyp, refs in zip(hyps, refs_list):
        htok = str(hyp).lower().split() if pd.notna(hyp) else []
        total = 0.0
        for ni in range(1, n+1):
            hng = ngrams(htok, ni)
            rng = Counter()
            for r in refs: rng += ngrams(str(r).lower().split(), ni)
            allg = set(hng) | set(rng)
            if not allg: continue
            hv = np.array([hng.get(g,0)*idf(g) for g in allg])
            rv = np.array([rng.get(g,0)*idf(g) for g in allg])
            nh, nr = np.linalg.norm(hv), np.linalg.norm(rv)
            if nh>0 and nr>0: total += np.dot(hv,rv)/(nh*nr)
        scores.append(total/n)
    return scores

print(f'CIDEr test: {compute_cider_scores(["a man on a couch"],[["a person sitting on a sofa"]])[0]:.3f}')

In [ ]:
# CELL 6 — METEOR + ROUGE-L
from rouge_score import rouge_scorer as rouge_mod

def compute_meteor_scores(hyps, refs_list):
    scores = []
    for hyp, refs in zip(hyps, refs_list):
        ht = str(hyp).lower().split() if pd.notna(hyp) else []
        rts = [str(r).lower().split() for r in refs if pd.notna(r)] or [[]]
        try: s = max(meteor_score([rt], ht) for rt in rts)
        except: s = 0.0
        scores.append(s)
    return scores

def compute_rougeL_scores(hyps, refs_list):
    scorer = rouge_mod.RougeScorer(['rougeL'], use_stemmer=True)
    scores = []
    for hyp, refs in zip(hyps, refs_list):
        hs = str(hyp).lower() if pd.notna(hyp) else ''
        best = max(
            (scorer.score(str(r).lower(), hs)['rougeL'].fmeasure
             for r in refs if pd.notna(r)),
            default=0.0
        )
        scores.append(best)
    return scores

print('METEOR test:', compute_meteor_scores(['a man on a couch'],[['a person sitting on a sofa']])[0])
print('ROUGE-L test:', compute_rougeL_scores(['a man on a couch'],[['a person sitting on a sofa']])[0])

## 🔄 Compute All Metrics for All Models

In [ ]:
# CELL 7 — Main metric computation loop
METRIC_THRESHOLDS = {
    'bleu4':  {'low':0.10,'high':0.30},
    'cider':  {'low':0.40,'high':0.80},
    'meteor': {'low':0.15,'high':0.30},
    'rougeL': {'low':0.30,'high':0.55},
}

print('🔄 Computing metrics...')
for model in tqdm(MODELS, desc='Models'):
    col = MODEL_CAP_COLS[model]
    if col not in df.columns:
        print(f'  ⚠️  {model}: caption column missing, skipping')
        continue
    hyps = df[col].fillna('').tolist()

    # BLEU 1-4
    bleu = compute_bleu_scores(hyps, refs_list)
    for n in range(1,5): df[f'bleu{n}_{model}'] = bleu[f'bleu{n}']

    # CIDEr
    df[f'cider_{model}'] = compute_cider_scores(hyps, refs_list)

    # METEOR
    df[f'meteor_{model}'] = compute_meteor_scores(hyps, refs_list)

    # ROUGE-L
    df[f'rougeL_{model}'] = compute_rougeL_scores(hyps, refs_list)

    # Confidence bins (based on BLEU-4)
    def conf_bin(s):
        t = METRIC_THRESHOLDS['bleu4']
        return 'low' if s < t['low'] else ('high' if s >= t['high'] else 'medium')
    df[f'bleu4_conf_{model}'] = df[f'bleu4_{model}'].apply(conf_bin)

    print(f'  {model.upper()}: BLEU4={df[f"bleu4_{model}"].mean():.4f}  '
          f'CIDEr={df[f"cider_{model}"].mean():.4f}  '
          f'METEOR={df[f"meteor_{model}"].mean():.4f}')

print('\n✅ All metrics computed.')

In [ ]:
# CELL 8 — Summary table
rows = []
for model in MODELS:
    row = {'Model': cfg.MODEL_DISPLAY.get(model, model.upper())}
    for metric in ['bleu1','bleu4','cider','meteor','rougeL']:
        c = f'{metric}_{model}'
        row[metric.upper()] = f'{df[c].mean():.4f} ± {df[c].std():.4f}' if c in df.columns else 'N/A'
    rows.append(row)

summary = pd.DataFrame(rows).set_index('Model')
print('\n📊 Metric Summary (mean ± std):')
print(summary.to_string())
summary.to_csv(OUTPUT_DIR/'metric_summary.csv')
print('\n💾 metric_summary.csv saved')

In [ ]:
# CELL 9 — BLEU-4 distribution plots
fig, axes = plt.subplots(1, len(MODELS), figsize=(4*len(MODELS), 4))
fig.suptitle('BLEU-4 Score Distribution per Model', fontsize=13, fontweight='bold')
colors = sns.color_palette('Set2', len(MODELS))

for ax, model, color in zip(axes, MODELS, colors):
    col = f'bleu4_{model}'
    if col not in df.columns: continue
    sc = df[col].dropna()
    ax.hist(sc, bins=20, color=color, edgecolor='white', alpha=0.85)
    ax.axvline(sc.mean(), color='red', lw=2, label=f'μ={sc.mean():.3f}')
    ax.axvline(BLEU_HIGH_THRESHOLD, color='black', lw=1.5,
               linestyle='--', label=f'threshold={BLEU_HIGH_THRESHOLD}')
    ax.set_title(cfg.MODEL_DISPLAY.get(model, model.upper()), fontweight='bold')
    ax.set_xlabel('BLEU-4'); ax.set_ylabel('Count')
    ax.legend(fontsize=8); ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
fig.savefig(FIGURES_DIR/'bleu4_distribution.png', dpi=150, bbox_inches='tight')
plt.show(); print('📊 Saved.')

In [ ]:
# CELL 10 — Cross-metric correlation heatmaps
fig, axes = plt.subplots(1, len(MODELS), figsize=(5*len(MODELS), 5))
fig.suptitle('Cross-Metric Correlation per Model', fontsize=13, fontweight='bold')

for ax, model in zip(axes, MODELS):
    cols = [f'{m}_{model}' for m in ['bleu1','bleu4','cider','meteor','rougeL']]
    avail = [c for c in cols if c in df.columns]
    if not avail: continue
    corr = df[avail].corr()
    corr.index = corr.columns = [c.replace(f'_{model}','') for c in avail]
    sns.heatmap(corr, ax=ax, annot=True, fmt='.2f', cmap='RdYlGn',
                center=0, vmin=-1, vmax=1, square=True, linewidths=0.5)
    ax.set_title(cfg.MODEL_DISPLAY.get(model, model.upper()), fontweight='bold')

plt.tight_layout()
fig.savefig(FIGURES_DIR/'cross_metric_correlation.png', dpi=150, bbox_inches='tight')
plt.show(); print('📊 Saved.')

In [ ]:
# CELL 11 — Box plots across models
metrics_plot = ['bleu4','cider','meteor','rougeL']
fig, axes = plt.subplots(1, len(metrics_plot), figsize=(4*len(metrics_plot), 5))
fig.suptitle('Metric Score Distributions Across Models', fontsize=13, fontweight='bold')

for ax, metric in zip(axes, metrics_plot):
    data = {cfg.MODEL_DISPLAY.get(m, m): df[f'{metric}_{m}'].dropna().values
            for m in MODELS if f'{metric}_{m}' in df.columns}
    if not data: continue
    bp = ax.boxplot(list(data.values()), labels=list(data.keys()),
                    patch_artist=True, notch=True)
    for patch, color in zip(bp['boxes'], sns.color_palette('Set2', len(data))):
        patch.set_facecolor(color); patch.set_alpha(0.8)
    ax.set_title(metric.upper(), fontweight='bold')
    ax.set_ylabel('Score'); ax.tick_params(axis='x', rotation=15)
    ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
fig.savefig(FIGURES_DIR/'metric_boxplots.png', dpi=150, bbox_inches='tight')
plt.show(); print('📊 Saved.')

In [ ]:
# CELL 12 — Save final metrics DataFrame
out_path = OUTPUT_DIR / 'metrics_all_models.csv'
df.to_csv(out_path, index=False)
m_cols = [c for c in df.columns if any(c.startswith(m) for m in ['bleu','cider','meteor','rougeL'])]
print(f'💾 metrics_all_models.csv saved: {out_path}')
print(f'   Metric columns added: {len(m_cols)}')
print('\n✅ Notebook 3 COMPLETE — run 04_human_annotation.ipynb next')